# Plot multi-lead predictions for predicting hurricane track (distance) errors.
author: Elizabeth A. Barnes and Randal J. Barnes

In [1]:
%matplotlib inline
%load_ext autotime

import sys
import importlib as imp
import warnings
from shapely.errors import ShapelyDeprecationWarning

warnings.filterwarnings("ignore", category=ShapelyDeprecationWarning)

import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import cartopy as ct
import plots
import compute_predictions

import experiment_settings
import mahalanobis

time: 1.55 s (started: 2023-06-08 10:43:54 -06:00)


In [27]:
__author__ = "Randal J Barnes and Elizabeth A. Barnes"
__version__ = "16 December 2022"

experiment = 'default_2023' # 'single_NN12_2023'

DATA_PATH = "data/"
FIGURE_PATH = "../track_martin/figures/"+experiment+"/analysis"
PREDICTIONS_PATH = "../track_martin/saved_predictions/"+experiment

time: 823 µs (started: 2023-06-08 11:59:54 -06:00)


In [28]:
plt.style.use("seaborn-white")
mpl.rcParams['savefig.dpi'] = 600
mpl.rcParams["figure.dpi"] = 100
dpiFig = 600
plots.set_plot_rc()
warnings.filterwarnings("ignore", category=np.VisibleDeprecationWarning)

time: 2.67 ms (started: 2023-06-08 11:59:54 -06:00)


/var/folders/1q/d4x36lf161717b4rhbv9m6s80000gq/T/ipykernel_857/2285215746.py:1: MatplotlibDeprecationWarning: The seaborn styles shipped by Matplotlib are deprecated since 3.6, as they no longer correspond to the styles shipped by seaborn. However, they will remain available as 'seaborn-v0_8-<style>'. Alternatively, directly use the seaborn API instead.
  plt.style.use("seaborn-white")


In [29]:
# EXP_NAME_VEC = (
#     # "centered_bivariate_normal_000_EPCP24",

#     "centered_bivariate_normal_100_EPCP12",
#     "centered_bivariate_normal_101_EPCP24",
#     "centered_bivariate_normal_102_EPCP36",
#     "centered_bivariate_normal_103_EPCP48",
#     "centered_bivariate_normal_104_EPCP60",
#     "centered_bivariate_normal_105_EPCP72",
#     "centered_bivariate_normal_106_EPCP84",
#     "centered_bivariate_normal_107_EPCP96",
#     "centered_bivariate_normal_108_EPCP108",
#     "centered_bivariate_normal_109_EPCP120",

#     "centered_bivariate_normal_200_AL12",
#     "centered_bivariate_normal_201_AL24",
#     "centered_bivariate_normal_202_AL36",
#     "centered_bivariate_normal_203_AL48",
#     "centered_bivariate_normal_204_AL60",
#     "centered_bivariate_normal_205_AL72",
#     "centered_bivariate_normal_206_AL84",
#     "centered_bivariate_normal_207_AL96",
#     "centered_bivariate_normal_208_AL108",
#     "centered_bivariate_normal_209_AL120",

#     )

EXP_NAME_VEC = (
    experiment+"_AL",
    experiment+"_EP",
    )

# EXP_NAME_VEC = (
#     "default_nolocation_EP12",
#     "default_nolocation_EP24",
#     "default_nolocation_EP36",
#     "default_nolocation_EP48",
#     "default_nolocation_EP60",
#     "default_nolocation_EP72",
#     "default_nolocation_EP84",
#     "default_nolocation_EP96",
#     "default_nolocation_EP108",
#     "default_nolocation_EP120",

#     "default_nolocation_AL12",
#     "default_nolocation_AL24",
#     "default_nolocation_AL36",
#     "default_nolocation_AL48",
#     "default_nolocation_AL60",
#     "default_nolocation_AL72",
#     "default_nolocation_AL84",
#     "default_nolocation_AL96",
#     "default_nolocation_AL108",
#     "default_nolocation_AL120",

#     )

time: 2.6 ms (started: 2023-06-08 11:59:54 -06:00)


# Plot Results

In [30]:
storm_dict = {
    "IAN": {"storm_name": "IAN",
            "year": 2022,
            "extent": [-100,-65,5,35],
            "nhc_cone_radius": {0:8, 12:26, 24:39, 36:52, 48:67, 60:84, 72:100, 96:142, 120:200}
            # https://www.nhc.noaa.gov/aboutcone.shtml
            },
    "FIONA": {"storm_name": "FIONA",
              "year": 2022,
              "extent": [-100,-45,10,55],
              "nhc_cone_radius": {0:8, 12:26, 24:39, 36:52, 48:67, 60:84, 72:100, 96:142, 120:200}
              },
    "IRMA": {"storm_name": "IRMA",
             "year": 2017,
             "extent": [-100,-20,10,40],
             "nhc_cone_radius": {0:8, 12:29, 24:45, 36:63, 48:78, 60:107, 72:107, 96:159, 120:211}
             # https://www.air-worldwide.com/blog/posts/2017/8/the-ever-shrinking-cone-of-uncertainty/
             },
    # "NICOLE": {"storm_name": "NICOLE",
    #            "year": 2022,
    #            "extent": [-100,-48,20,45],
    #            "nhc_cone_radius": {0:8, 12:26, 24:39, 36:52, 48:67, 60:84, 72:100, 96:142, 120:200}
    #            },
    # "JULIA": {"storm_name": "JULIA",
    #           "year": 2022,
    #           "extent": [-100,-65,5,20],
    #           "nhc_cone_radius": {0:8, 12:26, 24:39, 36:52, 48:67, 60:84, 72:100, 96:142, 120:200}
    #           },
    # "NORMAN": {"storm_name": "NORMAN",
    #             "year": 2018,
    #             "pred_time": 90306,
    #             "extent": [195, 360-135, 5, 35],
    #             "nhc_cone_radius": {0:8, 12:25, 24:40, 36:51, 48:66, 60:93, 72:93, 96:116, 120:151}
    #             },
    "HARVEY": {"storm_name": "HARVEY",
                "year": 2017,
                "extent": [-100, -25, 10, 55],
                "nhc_cone_radius": {0:8, 12:29, 24:45, 36:63, 48:78, 60:107, 72:107, 96:159, 120:211}
                },
    "DORIAN": {"storm_name": "DORIAN",
                "year": 2019,
                "extent": [-100, -25, 10, 55],
                "nhc_cone_radius": {0:8, 12:26, 24:41, 36:54, 48:68, 60:102, 72:102, 96:151, 120:198}
                },
}

time: 3.67 ms (started: 2023-06-08 11:59:55 -06:00)


In [31]:
imp.reload(mahalanobis)
imp.reload(plots)
imp.reload(compute_predictions)
import glob
KM_TO_DEG = 1.0 / 111.

for storm_name in ("FIONA","IAN","IRMA","HARVEY","DORIAN"):#("IAN","IRMA","FIONA"):#("IAN", "NICOLE", "IRMA"):
    print(storm_name)
    storm = storm_dict[storm_name]

    for RNG_SEED in (123,):

        TESTING_YEAR = storm["year"]
        files = glob.glob(PREDICTIONS_PATH + '/*')
        
        # GET PREDICTIONS
        df_pred_test = pd.DataFrame()
        for file in files:
            try:
                df = pd.read_csv(file)
            except:
                continue

            rng_seed = RNG_SEED
            years_test = (TESTING_YEAR,)
            df["exp_name"] = experiment
            df_pred_test = pd.concat([df_pred_test, df], axis=0)

        #------------------------------------------------------------
        # MAKE THE PLOTS
        df = df_pred_test.loc[
            (df_pred_test["Name"] == storm["storm_name"]) & (df_pred_test["year"] == TESTING_YEAR)].copy()
        df = df.sort_values("time").reset_index(drop=True)
        forecast_dates = df["time"].unique()

        for i,pred_time in enumerate(forecast_dates):
            print(str(i+1) + ' of ' + str(len(forecast_dates)) + ': ' + str(pred_time))
            storm["pred_time"] = pred_time
            df_storm = df_pred_test.loc[
                (df_pred_test["Name"] == storm["storm_name"]) & (df_pred_test["time"] == storm["pred_time"])].copy()
            df_storm = compute_predictions.add_lead_zero(df_storm)
            df_storm["nhc_cone_radius"] = [storm["nhc_cone_radius"][key] for key in df_storm["ftime(hr)"].unique()]
            
            # get dynamic extent
            extx = [df_storm["LONN"], df_storm["LONN"] + KM_TO_DEG * df_storm["OFDX"]]
            exty = [df_storm["LATN"], df_storm["LATN"] + KM_TO_DEG * df_storm["OFDY"]]
            storm_extent = [-(360-np.min(extx)+10), -(360-np.max(extx)-10), np.min(exty)-10, np.max(exty)+10]
            
            # plot probability ellipses
            fig = plt.figure(dpi=150, )
            ax = fig.add_subplot(1, 1, 1, projection=ct.crs.PlateCarree(central_longitude=0.))
            details = plots.plot_probability_ellipses(
                df_storm,
                ax=ax,
                leadtimes=np.arange(0,120+12, 12),
                contours=(.1, .25, .5, .75, .9,),
                # extent = storm["extent"],
                extent = storm_extent,
                alpha=.4,
                vector=True,
                plot_nhc_cone=True,
            )
            plt.gca().get_legend().remove()
            # ax.set_extent(storm["extent"], crs=ct.crs.PlateCarree())
            ax.set_extent(storm_extent, crs=ct.crs.PlateCarree())
            plt.savefig(
                FIGURE_PATH + '/probability_ellipses_rng_seed_' + str(RNG_SEED) + '_' + details.replace(' ', '_') + '.png',
                dpi=dpiFig,
                bbox_inches='tight',
            )
            plt.close()

            # plot banana cones
            try:
                fig = plt.figure(dpi=150, )
                ax = fig.add_subplot(1, 1, 1, projection=ct.crs.PlateCarree(central_longitude=0.))
                details = plots.plot_banana_of_uncertainty(
                    df_storm=df_storm,
                    ax=ax,
                    # extent=storm["extent"],
                    extent=storm_extent,
                    vector=True,
                    colors=("steelblue","khaki"),
                    alpha=.75,
                    plot_nhc_cone=True,
                )
                # ax.set_extent(storm["extent"], crs=ct.crs.PlateCarree())
                ax.set_extent(storm_extent, crs=ct.crs.PlateCarree())
                plt.savefig(
                    FIGURE_PATH + '/banana_cone_rng_seed_' + str(RNG_SEED) + '_' + details.replace(' ', '_') + '.png',
                    dpi=dpiFig,
                    bbox_inches='tight',
                )
                plt.close()
            except:
                print('not enough data for spline computation. not making the figure.')
                plt.close()

FIONA
1 of 35: 91412
2 of 35: 91418
3 of 35: 91500
4 of 35: 91506
5 of 35: 91512
6 of 35: 91518
7 of 35: 91600
8 of 35: 91606
9 of 35: 91612
10 of 35: 91618
11 of 35: 91700
12 of 35: 91706
13 of 35: 91712
14 of 35: 91718
15 of 35: 91800
16 of 35: 91806
17 of 35: 91812
18 of 35: 91818
19 of 35: 91900
20 of 35: 91906
21 of 35: 91912
22 of 35: 91918
23 of 35: 92000
24 of 35: 92006
25 of 35: 92012
26 of 35: 92018
27 of 35: 92100
28 of 35: 92106
29 of 35: 92118
30 of 35: 92200
31 of 35: 92206
32 of 35: 92212
not enough data for spline computation. not making the figure.
33 of 35: 92218
not enough data for spline computation. not making the figure.
34 of 35: 92300
not enough data for spline computation. not making the figure.
35 of 35: 92306
not enough data for spline computation. not making the figure.
IAN
1 of 29: 92306
2 of 29: 92312
3 of 29: 92318
4 of 29: 92400
5 of 29: 92406
6 of 29: 92412
7 of 29: 92418
8 of 29: 92500
9 of 29: 92506
10 of 29: 92512
11 of 29: 92518
12 of 29: 92600
13 o

In [19]:
# imp.reload(mahalanobis)
# imp.reload(plots)
# imp.reload(compute_predictions)
# KM_TO_DEG = 1.0 / 111.

# for storm_name in ("FIONA","IAN","IRMA","HARVEY","DORIAN"):#("IAN","IRMA","FIONA"):#("IAN", "NICOLE", "IRMA"):
#     print(storm_name)
#     storm = storm_dict[storm_name]

#     for RNG_SEED in (123,):

#         TESTING_YEAR = storm["year"]

#         # GET PREDICTIONS
#         df_pred_test = pd.DataFrame()
#         for exp_name in EXP_NAME_VEC:
#             # settings = experiment_settings.get_settings(exp_name)
            
#             # Create the model name.
#             model_name = (
#                     exp_name
#                     + "_"
#                     + str(TESTING_YEAR)
#                     + "_"
#                     + "centered_bivariate_normal"
#                     + "_"
#                     + f"rng_seed_{RNG_SEED}"
#                     # + "_"
#                     # + "OF"
#             )
#             try:
#                 prediction_filename = PREDICTIONS_PATH + '/' + model_name + "_testing_predictions.csv"
#                 print(prediction_filename)
#                 df = pd.read_csv(prediction_filename)
#             except:
#                 continue

#             rng_seed = RNG_SEED
#             years_test = (TESTING_YEAR,)
#             df["exp_name"] = exp_name
#             df_pred_test = pd.concat([df_pred_test, df], axis=0)

#         #------------------------------------------------------------
#         # MAKE THE PLOTS
#         df = df_pred_test.loc[
#             (df_pred_test["Name"] == storm["storm_name"])].copy()
#         df = df.sort_values("time").reset_index(drop=True)
#         forecast_dates = df["time"].unique()

#         for i,pred_time in enumerate(forecast_dates):
#             print(str(i+1) + ' of ' + str(len(forecast_dates)) + ': ' + str(pred_time))
#             storm["pred_time"] = pred_time
#             df_storm = df_pred_test.loc[
#                 (df_pred_test["Name"] == storm["storm_name"]) & (df_pred_test["time"] == storm["pred_time"])].copy()
#             df_storm = compute_predictions.add_lead_zero(df_storm)
#             df_storm["nhc_cone_radius"] = [storm["nhc_cone_radius"][key] for key in df_storm["ftime(hr)"].unique()]
            
#             # get dynamic extent
#             extx = [df_storm["LONN"], df_storm["LONN"] + KM_TO_DEG * df_storm["OFDX"]]
#             exty = [df_storm["LATN"], df_storm["LATN"] + KM_TO_DEG * df_storm["OFDY"]]
#             storm_extent = [-(360-np.min(extx)+5), -(360-np.max(extx)-5), np.min(exty)-5, np.max(exty)+5]
            
#             # plot probability ellipses
#             fig = plt.figure(dpi=150, )
#             ax = fig.add_subplot(1, 1, 1, projection=ct.crs.PlateCarree(central_longitude=0.))
#             details = plots.plot_probability_ellipses(
#                 df_storm,
#                 ax=ax,
#                 leadtimes=np.arange(0,120+12, 12),
#                 contours=(.1, .25, .5, .75, .9,),
#                 # extent = storm["extent"],
#                 extent = storm_extent,
#                 alpha=.4,
#                 vector=True,
#             )
#             plt.gca().get_legend().remove()
#             # ax.set_extent(storm["extent"], crs=ct.crs.PlateCarree())
#             ax.set_extent(storm_extent, crs=ct.crs.PlateCarree())
#             plt.savefig(
#                 FIGURE_PATH + '/probability_ellipses_rng_seed_' + str(RNG_SEED) + '_' + details.replace(' ', '_') + '.png',
#                 dpi=dpiFig,
#                 bbox_inches='tight',
#             )
#             plt.close()

#             # plot banana cones
#             try:
#                 fig = plt.figure(dpi=150, )
#                 ax = fig.add_subplot(1, 1, 1, projection=ct.crs.PlateCarree(central_longitude=0.))
#                 details = plots.plot_banana_of_uncertainty(
#                     df_storm=df_storm,
#                     ax=ax,
#                     # extent=storm["extent"],
#                     extent=storm_extent,
#                     vector=True,
#                     colors=("steelblue","khaki"),
#                     alpha=.75,
#                     plot_nhc_cone=True,
#                 )
#                 # ax.set_extent(storm["extent"], crs=ct.crs.PlateCarree())
#                 ax.set_extent(storm_extent, crs=ct.crs.PlateCarree())
#                 plt.savefig(
#                     FIGURE_PATH + '/banana_cone_rng_seed_' + str(RNG_SEED) + '_' + details.replace(' ', '_') + '.png',
#                     dpi=dpiFig,
#                     bbox_inches='tight',
#                 )
#                 plt.close()
#             except:
#                 print('not enough data for spline computation. not making the figure.')
#                 plt.close()

time: 4.82 ms (started: 2023-06-08 11:33:30 -06:00)
